## Notebook grammar
`setup → config → construct → aaaa-context → sweep → score → visualize → report`

Open in Colab: https://colab.research.google.com/github/HNXJ/jaxfne/blob/main/tutorials/jaxfne_v040_homeostasis_plasticity_dc_noise_sweep.ipynb

# v0.4.0 · Homeostasis, Plasticity & Drive/Noise Exponential Sweep

**Scope:** proxy scaffold — not calibrated physics.

Sweeps homeostasis global parameters (k_gain, r_star, tau_r_ms, alpha, eta) and  
noise/DC drive (5.0 · eⁿ) while running the **AAAA paradigm context**  
(L4 E neurons driven by periodic stimulus pulses).  
Homeostasis and plasticity are active throughout all sweep conditions.  
Output: similarity_pct per point, best-params table, heatmap + line plots.


In [ ]:
import importlib.util, subprocess, sys
from pathlib import Path
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "jaxfne").is_dir() and (_candidate / "pyproject.toml").exists():
        sys.path.insert(0, str(_candidate))
        break
if importlib.util.find_spec("jaxfne") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "jaxfne[viz,optax]"])


In [ ]:
import jax, jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd, json
from pathlib import Path
from types import SimpleNamespace
import jaxfne as jtfne
from jaxfne.tutorial_utils import (
    make_laminar_column_config, build_laminar_column,
    simulate_laminar_trials, spectrolaminar_from_trials,
)

jtfne.enable_x64()
print(f"jaxfne {jtfne.__version__}   jax {jax.__version__}")
print("Devices:", jax.devices())


## Scope — Computational Scaffold & Truth Gates

| Gate | Value |
|---|---|
| claim_level | computational_scaffold |
| field_solver_status | linear_solver |
| physical_amplitude_calibrated | False |

All fields are **`*_proxy`** readouts.  Similarity scores are a *relative* diagnostic.


In [ ]:
# ── Editable parameters ────────────────────────────────────────────────────────
N_NEURONS   = 100          # neurons in the V1 column
N_TRIALS    = 10           # independent trials per sweep point
DURATION_MS = 500.0        # ms per trial
DT_MS       = 0.1          # timestep (ms)
SEED        = 42
N_CONTACTS  = 16
FREQ_MIN_HZ = 1.0; FREQ_MAX_HZ = 150.0; FREQ_COUNT = 64
AB_HZ       = (10.0, 25.0)   # alpha-beta band
GA_HZ       = (40.0, 150.0)  # gamma band
N_SWEEP_PTS = 7

# Exponential sweep: SWEEP_BASE * exp(n), n in [-2, +1]
SWEEP_BASE  = 5.0

# AAAA paradigm drive (L4 E target)
A_AMP       = 5.0   # stimulus amplitude (native Izhikevich current units)
A_ONSET_MS  = 50.0  # onset within 500ms window
A_DUR_MS    = 150.0 # duration of one A-slot pulse

# Base homeostasis params (held fixed unless being swept)
BASE_HOMEO = dict(r_star=0.05, tau_r_ms=300.0, alpha=1.0, k_gain=1.0,
                  g_min=-12.0, g_max=8.0, r_max=1.0, eta=0.0, tau_x_ms=100.0,
                  w_min=-10.0, w_max=10.0)

OUTPUT_DIR = Path("outputs/v040_sweep")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

n_steps = int(DURATION_MS / DT_MS)
_n_vals = np.linspace(-2.0, 1.0, N_SWEEP_PTS)
EXP_SWEEP = SWEEP_BASE * np.exp(_n_vals)

K_GAIN_SWEEP   = np.array([0.0, 0.25, 0.5, 1.0, 1.5, 2.0, 2.5])
R_STAR_SWEEP   = np.array([0.01, 0.02, 0.05, 0.10, 0.20, 0.50, 1.0])
TAU_R_MS_SWEEP = np.array([50., 100., 200., 300., 500., 1000., 2000.])
ALPHA_SWEEP    = np.array([0.1,  0.3,  0.5,  1.0,  2.0,  3.0,  5.0])
ETA_SWEEP      = np.array([0.0, 0.001, 0.005, 0.01, 0.05, 0.10, 0.50])

print(f"n_steps={n_steps}   EXP_SWEEP≈{np.round(EXP_SWEEP,3)}")
print(f"Output: {OUTPUT_DIR}")


## 1 · Build tutorial-utils column (noise & drive sweeps)

Used by `simulate_laminar_trials` which accepts per-neuron Izhikevich overrides  
and an explicit drive schedule into `target_cells`.


In [ ]:
cfg_tu = make_laminar_column_config(
    areas=("V1",), n_neuron_per_column=N_NEURONS, n_trials=N_TRIALS,
    duration_ms=DURATION_MS, dt_ms=DT_MS, seed=SEED,
    n_contacts=N_CONTACTS, freq_min_hz=FREQ_MIN_HZ,
    freq_max_hz=FREQ_MAX_HZ, freq_count=FREQ_COUNT,
    output_dir=str(OUTPUT_DIR),
)
model_tu = build_laminar_column(cfg_tu)
ndf = model_tu["neurons"]

# L4 E neuron indices (stimulated by the AAAA paradigm)
l4e_tu = np.flatnonzero((ndf["cell_type"] == "E") & (ndf["layer"] == "L4"))
print(f"Tutorial-utils column: {len(ndf)} neurons   L4 E count: {len(l4e_tu)}")

# AAAA drive: one rectangular pulse on L4 E neurons
_t = np.arange(n_steps) * DT_MS
aaaa_stim = np.where((_t >= A_ONSET_MS) & (_t < A_ONSET_MS + A_DUR_MS),
                      A_AMP, 0.0).astype(np.float32)
print(f"AAAA stimulus: amplitude={A_AMP}  onset={A_ONSET_MS}ms  dur={A_DUR_MS}ms  "
      f"nonzero steps={int((aaaa_stim>0).sum())}")


## 2 · Spectrolaminar similarity helpers

In [ ]:
def _score_trials(trials_dict, cfg) -> float:
    """Spectrolaminar motif similarity_pct from a trials dict (0=noise, 100=ideal motif)."""
    _, spec = spectrolaminar_from_trials(
        trials_dict, cfg, signal_key="csd_contacts",
        freq_min_hz=FREQ_MIN_HZ, freq_max_hz=FREQ_MAX_HZ, freq_count=FREQ_COUNT,
        alpha_beta_range_hz=AB_HZ, gamma_range_hz=GA_HZ, area_index=0, area_name="V1",
    )
    ab = np.asarray(spec["alpha_beta"], dtype=np.float64)
    gm = np.asarray(spec["gamma"],      dtype=np.float64)
    if ab.size < 2 or np.std(ab) < 1e-9 or np.std(gm) < 1e-9:
        return 50.0
    corr = float(np.corrcoef(ab, gm)[0, 1])
    return float(np.clip((1.0 - corr) * 50.0, 0.0, 100.0))


def _run_tu(izh_override: dict) -> float:
    """Run N_TRIALS via tutorial-utils path with AAAA drive on L4 E neurons."""
    from dataclasses import replace
    cfg_s = replace(cfg_tu, cell_type_izh_params=izh_override)
    trials = simulate_laminar_trials(
        model_tu, cfg_s, n_trials=N_TRIALS,
        stimulus=aaaa_stim, target_cells=l4e_tu,
    )
    return _score_trials(trials, cfg_s)


print("Tutorial-utils helpers ready.")


## 3 · Build Config-path model (homeostasis & plasticity sweeps)

`build_laminar_column` → `construct` → `simulate` with  
`RuntimeConfig(enable_homeostasis=True, homeostasis_params={...})`.  
L4 E neurons targeted via `StimulusSchedule(target_indices=...)`.


In [ ]:
cfg_cp = (
    jtfne.build_laminar_column(name="V1", n=N_NEURONS, ei_profile="canonical")
    .runtime(seed=SEED, recurrent_backend="edge_list")
    .set_emitter("izhikevich", "cortical_eig")
    .probes(["spikes", "CSD-proxy"], n_contacts=N_CONTACTS)
    .field(domain="laminar_column", conductivity="proxy",
           boundary="mean_zero_neumann")
)
model_cp = jtfne.construct(cfg_cp)
nt = model_cp.neuron_table()
N_TOTAL = len(nt)

# L4 E indices in Config-path model
l4e_cp = [i for i, r in enumerate(nt)
          if r.get("layer") in ("L4",) and r.get("cell_type") == "E"]
print(f"Config-path model: {N_TOTAL} neurons   L4 E: {len(l4e_cp)}")

# AAAA StimulusSchedule targeting L4 E neurons
aaaa_sched = jtfne.StimulusSchedule(
    events=(
        {"onset_ms": A_ONSET_MS, "duration_ms": A_DUR_MS,
         "amplitude": A_AMP, "label": "A1",
         "is_drive_event": True, "target_indices": l4e_cp},
    ),
    n_neurons=N_TOTAL,
)

col_h = 1.6e-3  # 1.6 mm canonical column height

def _score_cp(hp_override: dict) -> float:
    """Run N_TRIALS via Config-path with AAAA drive + homeostasis+plasticity."""
    hp = {**BASE_HOMEO, **hp_override}
    rt = jtfne.RuntimeConfig(
        recurrent_backend="edge_list",
        enable_homeostasis=True,
        homeostasis_params=hp,
    )
    contacts = []
    for i in range(N_TRIALS):
        sig = jtfne.simulate(model_cp,
            sim=jtfne.Simulation(duration_ms=DURATION_MS, dt_ms=DT_MS,
                                  seed=SEED + i, runtime=rt),
            paradigm=aaaa_sched,
        )
        contacts.append(np.asarray(sig.field.csd_proxy))  # (T, n_contacts)
    arr = np.stack(contacts)  # (N_TRIALS, T, n_contacts)
    n_c = arr.shape[-1]
    cfg_ns = SimpleNamespace(
        dt_ms=DT_MS, duration_ms=DURATION_MS, freq_min_hz=FREQ_MIN_HZ,
        freq_max_hz=FREQ_MAX_HZ, freq_count=FREQ_COUNT, l4_ref_rel=0.5,
        cz_m=col_h, output_dir=None, n_trials=N_TRIALS, areas=("V1",),
    )
    td = {"csd_contacts": arr,
          "contact_depths_m": np.linspace(0.0, col_h, n_c)}
    return _score_trials(td, cfg_ns)

# JIT warmup (k_gain=1.0, eta=0.0, AAAA drive)
print("JIT warmup (first compile) ...")
_w = _score_cp({})
print(f"Warmup similarity = {_w:.1f}%   (N_compile should be 1 after this)")


## 4 · Sweep: noise amplitude  `5.0 · eⁿ`

Per-neuron internal noise coefficient for all cell types.  
Default noise_scale = 0.5.  AAAA drive active; homeostasis ON (k_gain=1.0).


In [ ]:
print("Sweep: NOISE (5.0·eⁿ) ...")
noise_scores = []
for nv in EXP_SWEEP:
    izh = {ct: {"noise": float(nv)} for ct in ("E","PV","SST","VIP")}
    s = _run_tu(izh)
    noise_scores.append(s)
    print(f"  noise={nv:.4f}  sim={s:.1f}%")
bi = int(np.argmax(noise_scores))
print(f"  → best noise={EXP_SWEEP[bi]:.4f}  sim={noise_scores[bi]:.1f}%")


## 5 · Sweep: DC drive amplitude  `5.0 · eⁿ`  (L4 E neurons)

In [ ]:
print("Sweep: DC DRIVE (5.0·eⁿ) ...")
drive_scores = []
for dv in EXP_SWEEP:
    # Scale stimulus amplitude AND change base drive for E neurons
    izh = {"E": {"drive": float(dv)}}
    _t2 = np.arange(n_steps) * DT_MS
    stim_dv = np.where((_t2 >= A_ONSET_MS) & (_t2 < A_ONSET_MS + A_DUR_MS),
                        dv, 0.0).astype(np.float32)
    from dataclasses import replace as _replace
    cfg_s2 = _replace(cfg_tu, cell_type_izh_params=izh)
    trials = simulate_laminar_trials(model_tu, cfg_s2, n_trials=N_TRIALS,
                                      stimulus=stim_dv, target_cells=l4e_tu)
    s = _score_trials(trials, cfg_s2)
    drive_scores.append(s)
    print(f"  drive={dv:.4f}  sim={s:.1f}%")
bi = int(np.argmax(drive_scores))
print(f"  → best drive={EXP_SWEEP[bi]:.4f}  sim={drive_scores[bi]:.1f}%")


## 6 · Sweep: Homeostasis k_gain

`g_i = clip(k_gain·(r_star − r_i), g_min, g_max)`.  k_gain=0 → no homeostasis.  
AAAA drive active on L4 E neurons throughout.


In [ ]:
print("Sweep: K_GAIN ...")
kgain_scores = []
for k in K_GAIN_SWEEP:
    s = _score_cp({"k_gain": float(k)})
    kgain_scores.append(s)
    print(f"  k_gain={k:.3f}  sim={s:.1f}%")
bi = int(np.argmax(kgain_scores))
print(f"  → best k_gain={K_GAIN_SWEEP[bi]:.3f}  sim={kgain_scores[bi]:.1f}%")


## 7 · Sweep: Homeostasis r_star (target activity trace)

In [ ]:
print("Sweep: R_STAR ...")
rstar_scores = []
for rs in R_STAR_SWEEP:
    s = _score_cp({"r_star": float(rs)})
    rstar_scores.append(s)
    print(f"  r_star={rs:.4f}  sim={s:.1f}%")
bi = int(np.argmax(rstar_scores))
print(f"  → best r_star={R_STAR_SWEEP[bi]:.4f}  sim={rstar_scores[bi]:.1f}%")


## 8 · Sweep: Homeostasis tau_r_ms (slow-leak timescale)

In [ ]:
print("Sweep: TAU_R_MS ...")
tau_scores = []
for tau in TAU_R_MS_SWEEP:
    s = _score_cp({"tau_r_ms": float(tau)})
    tau_scores.append(s)
    print(f"  tau_r_ms={tau:.1f}  sim={s:.1f}%")
bi = int(np.argmax(tau_scores))
print(f"  → best tau_r_ms={TAU_R_MS_SWEEP[bi]:.1f}  sim={tau_scores[bi]:.1f}%")


## 9 · Sweep: Homeostasis alpha (per-spike activity-trace jump)

In [ ]:
print("Sweep: ALPHA ...")
alpha_scores = []
for alp in ALPHA_SWEEP:
    s = _score_cp({"alpha": float(alp)})
    alpha_scores.append(s)
    print(f"  alpha={alp:.2f}  sim={s:.1f}%")
bi = int(np.argmax(alpha_scores))
print(f"  → best alpha={ALPHA_SWEEP[bi]:.2f}  sim={alpha_scores[bi]:.1f}%")


## 10 · Sweep: Homeostatic synaptic plasticity — eta

`dw_{j→i} = eta·(r_star − r_i)·x_j`.  eta=0 → pure homeostasis (no plasticity).  
Plasticity co-runs with homeostasis; AAAA drive active on L4 E neurons.


In [ ]:
print("Sweep: ETA (plasticity) ...")
eta_scores = []
for ev in ETA_SWEEP:
    s = _score_cp({"eta": float(ev)})
    eta_scores.append(s)
    print(f"  eta={ev:.4f}  sim={s:.1f}%")
bi = int(np.argmax(eta_scores))
print(f"  → best eta={ETA_SWEEP[bi]:.4f}  sim={eta_scores[bi]:.1f}%")


## 11 · Summary: best parameters & heatmap

In [ ]:
sweep_rows = [
    ("noise_scale",  EXP_SWEEP,      noise_scores),
    ("E_drive",      EXP_SWEEP,      drive_scores),
    ("k_gain",       K_GAIN_SWEEP,   kgain_scores),
    ("r_star",       R_STAR_SWEEP,   rstar_scores),
    ("tau_r_ms",     TAU_R_MS_SWEEP, tau_scores),
    ("alpha",        ALPHA_SWEEP,    alpha_scores),
    ("eta",          ETA_SWEEP,      eta_scores),
]

best_rows = []
for param, vals, scores in sweep_rows:
    bi = int(np.argmax(scores))
    best_rows.append({
        "parameter":       param,
        "best_value":      float(vals[bi]),
        "best_similarity": float(scores[bi]),
        "mean_similarity": float(np.mean(scores)),
    })
df = pd.DataFrame(best_rows)
print(df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))


In [ ]:
# Heatmap + line-plot grid
fig, axes = plt.subplots(len(sweep_rows), 2, figsize=(13, 2.5*len(sweep_rows)),
                          constrained_layout=True)
fig.suptitle(
    "Spectrolaminar Motif Similarity — Proxy sweep under AAAA drive context\n"
    f"N={N_NEURONS} neurons · {N_TRIALS} trials · {DURATION_MS:.0f} ms · dt={DT_MS} ms · "
    "homeostasis+plasticity active",
    fontsize=11, y=1.01)

for row_i, (param, vals, scores) in enumerate(sweep_rows):
    bi = int(np.argmax(scores))
    ax_h, ax_l = axes[row_i]

    # Heatmap
    im = ax_h.imshow(np.array(scores)[None, :], aspect="auto",
                      cmap="viridis", vmin=0, vmax=100,
                      extent=[0, len(vals), -0.5, 0.5])
    ax_h.set_xticks(np.arange(len(vals)) + 0.5)
    ax_h.set_xticklabels([f"{v:.2g}" for v in vals], fontsize=7)
    ax_h.set_yticks([])
    ax_h.set_xlabel(param, fontsize=9)
    ax_h.axvline(bi + 0.5, color="white", lw=1.5, ls="--")
    plt.colorbar(im, ax=ax_h, fraction=0.04, pad=0.02, label="sim%")

    # Line plot
    ax_l.plot(vals, scores, "o-", lw=2, ms=5, color="steelblue")
    ax_l.axvline(vals[bi], color="tomato", lw=1.5, ls="--",
                  label=f"best={vals[bi]:.3g}")
    ax_l.set_xlabel(param, fontsize=9)
    ax_l.set_ylabel("similarity %", fontsize=8)
    ax_l.set_ylim(0, 105)
    ax_l.legend(fontsize=8); ax_l.grid(True, alpha=0.3)

fig.savefig(OUTPUT_DIR / "sweep_summary.png", dpi=120, bbox_inches="tight")
print("Saved: sweep_summary.png  (proxy readout — computational scaffold)")
plt.show()


## 12 · Manifest & validation report

In [ ]:
manifest = {
    "notebook": "jaxfne_v040_homeostasis_plasticity_dc_noise_sweep",
    "jaxfne_version": jtfne.__version__,
    "config": {"N_NEURONS": N_NEURONS, "N_TRIALS": N_TRIALS,
               "DURATION_MS": DURATION_MS, "DT_MS": DT_MS, "SEED": SEED,
               "SWEEP_BASE": SWEEP_BASE, "N_SWEEP_PTS": N_SWEEP_PTS,
               "A_AMP": A_AMP, "A_ONSET_MS": A_ONSET_MS, "A_DUR_MS": A_DUR_MS,
               "paradigm_context": "AAAA (L4 E driven)",
               "homeostasis_during_sweep": True,
               "plasticity_during_sweep": True},
    "sweep_results": {
        param: {"values": vals.tolist(), "scores": [float(s) for s in scores],
                "best_value": float(vals[int(np.argmax(scores))]),
                "best_similarity_pct": float(np.max(scores))}
        for param, vals, scores in sweep_rows
    },
    "claim_level": "computational_scaffold",
    "field_solver_status": "linear_solver",
    "physical_amplitude_calibrated": False,
}
with open(OUTPUT_DIR / "sweep_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

val = {
    "all_finite": all(np.isfinite(s) for _,_,sc in sweep_rows for s in sc),
    "all_in_range": all(0<=s<=100 for _,_,sc in sweep_rows for s in sc),
    "n_dims": len(sweep_rows), "n_pts": N_SWEEP_PTS,
}
with open(OUTPUT_DIR / "sweep_validation.json", "w") as f:
    json.dump(val, f, indent=2)
print("sweep_manifest.json + sweep_validation.json written.")
print("Validation:", val)
